# Pipeline Principal — Hopifield
Executa as etapas de pré-processamento e análise da matriz de expressão.

## 1. Binarização

In [1]:
import sys, os, importlib

SRC_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config, preprocessing
importlib.reload(config)
importlib.reload(preprocessing)
from config import PATH_M, PATH_F, OUT_BINARIZACAO
from preprocessing import Binarizador

# --- Binarização dos dois datasets ---
binarizador_m = Binarizador(path_h5ad=PATH_M, out_dir=OUT_BINARIZACAO)
binarizador_f = Binarizador(path_h5ad=PATH_F, out_dir=OUT_BINARIZACAO)

binarizador_m.binarizar()
binarizador_f.binarizar()

print("Mathys binarizado em:", binarizador_m.path_binarizada)
print("Fujita binarizado em:", binarizador_f.path_binarizada)

[Binarizador] Arquivo já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaM\matrizBinarizadaM.h5ad
[Binarizador] Arquivo já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaF\matrizBinarizadaM.h5ad
Mathys binarizado em: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaM\matrizBinarizadaM.h5ad
Fujita binarizado em: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaF\matrizBinarizadaM.h5ad


## 2. Alinhamento

In [2]:
import sys, os, importlib
import anndata as ad

SRC_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config, alinhamento
importlib.reload(config)
importlib.reload(alinhamento)
from config import PATH_FEATURES_F, PATH_FEATURES_M, PATH_TOP5000, OUT_ALINHAMENTO, OUT_TOP_GENES
from alinhamento import (LeitorFeatures, AnalisadorSobreposicao, Alinhador,
                          ValidadorAlinhamento, AnalisadorCobertura,
                          SelecionadorGenesFrequentes)

In [3]:
# Passo 1 — Leitura dos arquivos de features
leitor = LeitorFeatures(PATH_FEATURES_F, PATH_FEATURES_M)
leitor.ler()
print(leitor)

[LeitorFeatures] Fujita : 36591 genes mapeados
[LeitorFeatures] Mathys : 32643 genes mapeados
LeitorFeatures(
  path_features_f = C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataF\dados_combinados\features.tsv\features.tsv
  path_features_m = C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataM\featuresM.tsv.gz
  map_f           = 36591 genes
  map_m           = 32643 genes
)


In [4]:
# Passo 2 — Análise de sobreposição
from config import PATH_F

# var_names são idênticos no original e no binarizado — lemos direto do original
_f = ad.read_h5ad(PATH_F, backed='r')
var_names_f_original = _f.var_names.tolist()
_f.file.close()
del _f

analisador = AnalisadorSobreposicao(leitor.map_f, leitor.map_m, var_names_f_original)
analisador.analisar()
print(analisador)

[AnalisadorSobreposicao] Em comum  : 30312
[AnalisadorSobreposicao] Só Fujita : 6279
[AnalisadorSobreposicao] Só Mathys : 2331  ← serão excluídos
[AnalisadorSobreposicao] Espaço gênico final: 36601 genes
AnalisadorSobreposicao(
  ids_comuns      = 30312
  ids_so_f        = 6279
  ids_so_m        = 2331
  genes_ordenados = 36601
)


In [5]:
# Passo 3 — Alinhamento dos dois h5ad binarizados
alinhador = Alinhador(
    path_binarizada_m = binarizador_m.path_binarizada,
    path_binarizada_f = binarizador_f.path_binarizada,
    out_dir           = OUT_ALINHAMENTO,
    map_f             = leitor.map_f,
    map_m             = leitor.map_m,
    gene_alvo_idx     = analisador.gene_alvo_idx,
    genes_ordenados   = analisador.genes_ordenados,
)
alinhador.alinhar()
alinhador.salvar_como_txt()
alinhador.gerar_tracking(analisador.ids_so_f, leitor.map_f)
print(alinhador)

[Alinhador] Fujita já alinhado, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.h5ad
[Alinhador] Mathys já alinhado, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.h5ad

[Alinhador] Concluído.
[Alinhador] TXT já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
[Alinhador] TXT já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.txt
[Alinhador] Tracking já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\tracking_genes_adicionados_mathys.csv
Alinhador(
  path_binarizada_m = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\ma

In [6]:
# Passo 3b — Top genes mais frequentes do arquivo alinhado
path_top5000_out = os.path.join(OUT_TOP_GENES, "top5000_frequentes.csv")
if os.path.exists(path_top5000_out):
    print(f"[SKIP] top5000_frequentes.csv já existe: {path_top5000_out}")
else:
    selecionador = SelecionadorGenesFrequentes(
        path_txt = alinhador.path_f_alinhado.replace('.h5ad', '.txt'),
        n        = 5000,
    )
    selecionador.calcular()
    selecionador.salvar(path_top5000_out)
    print(selecionador)

[SelecionadorGenesFrequentes] Lendo: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
  Calculando frequências para 36600 genes...
[SelecionadorGenesFrequentes] Concluído. 40913 células, top 5000 genes selecionados.
[SelecionadorGenesFrequentes] Salvo em: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\top_genes\top5000_frequentes.csv
SelecionadorGenesFrequentes(
  path_txt     = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
  n            = 5000
  df_resultado = 5000 genes
)


In [7]:
import sys, os, importlib

SRC_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import treinamento
importlib.reload(treinamento)
from treinamento import GeradorConjuntoTreinamento
from config import OUT_TOP_GENES, OUT_TREINAMENTO

# Passo 6 — Conjuntos de treinamento para a rede Hopfield
gerador = GeradorConjuntoTreinamento(
    path_top_genes_csv = os.path.join(OUT_TOP_GENES, "top5000_frequentes.csv"),
    out_dir            = OUT_TREINAMENTO,
)

gerador.gerar(alinhador.path_f_alinhado.replace('.h5ad', '.txt'))  # Fujita
gerador.gerar(alinhador.path_m_alinhado.replace('.h5ad', '.txt'))  # Mathys

print(gerador)

[GeradorConjuntoTreinamento] 5000 genes carregados de: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\top_genes\top5000_frequentes.csv

[GeradorConjuntoTreinamento] Processando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
  Genes encontrados no arquivo: 5000 de 5000 selecionados
  Escrevendo via Polars streaming...
[GeradorConjuntoTreinamento] Salvo: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\treinamento\adataF_binarizado_alinhado_top5000.txt  (5000 genes)

[GeradorConjuntoTreinamento] Processando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.txt
  Genes encontrados no arquivo: 5000 de 5000 selecionados
  Escrevendo via Polars streaming...
[GeradorConjuntoTreinamento] Salvo: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\treinamento\adataM_binar

In [8]:
# Passo 4 — Validação da ordem de genes
validador = ValidadorAlinhamento(
    path_f_alinhado = alinhador.path_f_alinhado,
    path_m_alinhado = alinhador.path_m_alinhado,
    genes_ordenados = analisador.genes_ordenados,
)
validador.validar()

[ValidadorAlinhamento] Carregando metadados...
✓ Número de genes idêntico: 36601
✓ Fujita alinhado == ordem de referência
✓ Mathys alinhado == ordem de referência
✓ Fujita alinhado == Mathys alinhado
[ValidadorAlinhamento] Validação concluída com sucesso.


ValidadorAlinhamento(
  path_f_alinhado  = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.h5ad
  path_m_alinhado  = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.h5ad
  genes_ordenados  = 36601 genes
)

In [9]:
# Passo 5 — Cobertura dos top-5000 genes frequentes do Fujita no Mathys
cobertura = AnalisadorCobertura(PATH_TOP5000, leitor.map_f, leitor.map_m)
cobertura.analisar(out_csv=os.path.join(OUT_ALINHAMENTO, "top5000_cobertura_mathys.csv"))

[AnalisadorCobertura] Arquivo já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\top5000_cobertura_mathys.csv


gene_name,frequencia,ensembl_id,presente_mathys,sem_ensembl_fujita
str,i64,str,bool,bool
"""MALAT1""",40782,"""ENSG00000251562""",true,false
"""CADM2""",39065,"""ENSG00000175161""",true,false
"""PCDH9""",39012,"""ENSG00000184226""",true,false
"""SNHG14""",38439,"""ENSG00000224078""",false,false
"""DLG2""",38107,"""ENSG00000150672""",true,false
…,…,…,…,…
"""METTL26""",11033,"""ENSG00000130731""",true,false
"""NDUFAF5""",11031,"""ENSG00000101247""",true,false
"""CALU""",11030,"""ENSG00000128595""",true,false
